# Refresh Serving Tables from Pipeline Gold Views

This notebook syncs data from the Spark Declarative Pipeline (SDP) gold materialized views to serving Delta tables.

**Why separate serving tables?**
- The pipeline creates materialized views in a pipeline-managed schema
- The backend API queries Delta tables in the main schema
- This notebook bridges the gap by copying data from views to tables

**Tables Refreshed:**
- `shipments` - Active and delivered shipments
- `incidents` - Current and historical incidents
- `capacity_lanes` - Lane capacity and utilization

## Configuration

Load catalog and schema from job parameters, environment variables, or use defaults.

In [ ]:
import os


def _get_config(name: str, default: str) -> str:
    """Get config from job parameters (widgets), env vars, or default."""
    try:
        return dbutils.widgets.get(name)  # type: ignore[name-defined]
    except Exception:
        return os.environ.get(f"DATABRICKS_{name.upper()}", default)


CATALOG = _get_config("catalog", "main")
SCHEMA = _get_config("schema", "logistics_control_center")
GOLD = f"{CATALOG}.{SCHEMA}"

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Gold prefix: {GOLD}")

## Refresh Shipments Table

Copy shipment data from the pipeline's gold materialized view to the serving table.

In [ ]:
spark.sql(
    f"""
    INSERT OVERWRITE {CATALOG}.{SCHEMA}.shipments (trackingId, customerId, priority, laneId, promisedETA, currentETA, packageCount, status)
    SELECT
      trackingId,
      customerId,
      priority,
      laneId,
      promisedETA,
      currentETA,
      packageCount,
      status
    FROM {GOLD}.logistics_shipments_gold
    """
)

print(f"✓ Refreshed {CATALOG}.{SCHEMA}.shipments")

## Refresh Incidents Table

Copy incident data from the pipeline's gold materialized view to the serving table.

In [ ]:
spark.sql(
    f"""
    INSERT OVERWRITE {CATALOG}.{SCHEMA}.incidents (id, laneId, timestamp, type, ref, cause, impactMinutes, impactThroughputPct, confidence, active)
    SELECT
      id,
      laneId,
      timestamp,
      type,
      ref,
      cause,
      impactMinutes,
      impactThroughputPct,
      confidence,
      active
    FROM {GOLD}.logistics_incidents_gold
    """
)

print(f"✓ Refreshed {CATALOG}.{SCHEMA}.incidents")

## Refresh Capacity Lanes Table

Copy lane capacity data from the pipeline's gold materialized view to the serving table.

In [ ]:
spark.sql(
    f"""
    INSERT OVERWRITE {CATALOG}.{SCHEMA}.capacity_lanes (id, origin, dest, mode, avgDailyVolume, onTimePct, delayMinutes, slaRiskPct, maxCapacity, utilizationPct, availableCapacity, optimalUtilization)
    SELECT
      id,
      origin,
      dest,
      mode,
      avgDailyVolume,
      onTimePct,
      delayMinutes,
      slaRiskPct,
      maxCapacity,
      utilizationPct,
      availableCapacity,
      optimalUtilization
    FROM {GOLD}.logistics_capacity_gold
    """
)

print(f"✓ Refreshed {CATALOG}.{SCHEMA}.capacity_lanes")

## Summary

Confirm all serving tables have been refreshed.

In [ ]:
print("\n" + "="*50)
print("Serving tables refreshed from pipeline gold views.")
print("="*50)
print(f"  ✓ {CATALOG}.{SCHEMA}.shipments")
print(f"  ✓ {CATALOG}.{SCHEMA}.incidents")
print(f"  ✓ {CATALOG}.{SCHEMA}.capacity_lanes")